In [2]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [5]:
user_query = "hi"

In [6]:
intent_router_agent = Agent(
    role="Intent Router",
    goal="Classify the user query into greeting, transport query, or unrelated query.",
    backstory=(
        "You analyze user messages for the SmartMove transport assistant. "
        "You determine whether the message is a greeting, a bus transport query, "
        "or an unrelated question."
    ),
    llm=llm,
    verbose=True
)

intent_router_task = Task(
    description=f"""
User Query:
{user_query}

Classify the query into one of these categories:

1. GREETING
2. TRANSPORT_QUERY
3. IRRELEVANT_QUERY

Return only one label.
""",
    agent=intent_router_agent,
    expected_output="GREETING or TRANSPORT_QUERY or IRRELEVANT_QUERY"
)



In [12]:
greeting_agent = Agent(
    role="SmartMove Greeting Assistant",
    goal="Provide the official SmartMove welcome message.",
    backstory=(
        "You represent SmartMove, a Sri Lankan bus information assistant. "
        "When users greet the system, you respond with the SmartMove welcome message."
    ),
    llm=llm,
    verbose=True
)

greeting_task = Task(
    description=f"""
User Query:
{user_query}

Respond with the official SmartMove welcome message.

Rules:
- Keep it short
- Do not answer personal questions
- Always introduce SmartMove

Response example:

"Welcome to SmartMove. SmartMove provides information on bus fares and schedules in Sri Lanka. Please tell me your travel route."
""",
    agent=greeting_agent,
    expected_output="SmartMove welcome message",
    context=[intent_router_task]
)

In [ ]:
transport_agent = Agent(
    role="SmartMove Transport Assistant",
    goal="Extract transportation entities: origin, destination, time, date from queries",
    backstory=(
        "You are an expert at extracting structured information from natural language. "
        "You identify origin locations, destination locations, time preferences, and dates "
        "from user queries about transportation in Sri Lanka."
    ),
    llm=llm,
    verbose=True
)

transport_task = Task(
        description=(
            f"""Extract entities from this query: '''{user_query}'''
            - from_location: Origin/starting location (e.g., Colombo, Kandy)
            - to_location: Destination/ending location
            - time: Time preference (e.g., '6 pm', 'afternoon', 'morning')
            - date: Date if mentioned (e.g., 'tomorrow', 'today', specific date)
            Return ONLY valid JSON: {{\'from_location\': \"<location>\", \'to_location\': \"<location>\", 
            \'time\': \"<time>\", \'date\': \"<date>\"}}
            Use empty string '' for missing values.
            You MUST return ONLY valid JSON, no markdown, no explanations."""
        ),
        agent=transport_agent,
        expected_output="JSON with extracted entities",
        context=[intent_router_task]
    )

SyntaxError: f-string: valid expression required before '}' (1980324271.py, line 15)

In [ ]:
fallback_agent = Agent(
    role="SmartMove Scope Guard",
    goal="Politely inform users that SmartMove only provides transport information.",
    backstory=(
        "You ensure SmartMove stays within its scope. "
        "If a user asks about unrelated topics like weather, politics, or general knowledge, "
        "you politely inform them that SmartMove only supports bus schedule and fare queries."
    ),
    llm=llm,
    verbose=True
)

fallback_task = Task(
    description=f"""
User Query:
{user_query}

Respond with a polite message explaining that SmartMove only provides
bus schedule and fare information in Sri Lanka.

Response format:

"Sorry, I am not aware about '{user_query}'. SmartMove currently provides information about bus schedules and fares in Sri Lanka."
""",
    agent=fallback_agent,
    expected_output="Apology message",
    context=[intent_router_task]
)

In [ ]:
crew = Crew(
    agents=[
        intent_router_agent,
        greeting_agent,
        transport_agent,
        fallback_agent
    ],

    tasks=[
        intent_router_task,
        greeting_task,
        transport_task,
        fallback_task
    ],

    process=Process.sequential
)

In [14]:
result = crew.kickoff()

print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Intent Router                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  User Query:                                                                                                    │
│  hi                                                                                                             │
│                                                                                                                 │
│  Classify the query into one of these categories:                                                               │
│                                                                                                                 │
│  1. GREETING                                                                                                    │
│  2. TRANSPORT_QUERY                                                                                             │
│  3. IRRELEVANT_QUERY                                                                                            │
│                                                                                                                 │
│  Return only one label.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Intent Router                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  GREETING                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Greeting Assistant                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  User Query:                                                                                                    │
│  hi                                                                                                             │
│                                                                                                                 │
│  Respond with the official SmartMove welcome message.                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│  - Keep it short                                                                                                │
│  - Do not answer personal questions                                                                             │
│  - Always introduce SmartMove                                                                                   │
│                                                                                                                 │
│  Response example:                                                                                              │
│                                                                                                                 │
│  "Welcome to SmartMove. SmartMove provides information on bus fares and schedules in Sri Lanka. Please tell me  │
│  your travel route."                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Greeting Assistant                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Welcome to SmartMove. SmartMove provides information on bus fares and schedules in Sri Lanka. Please tell me   │
│  your travel route.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Transport Assistant                                                                           │
│                                                                                                                 │
│  Task: Extract entities from this query: '''hi'''                                                               │
│              - from_location: Origin/starting location (e.g., Colombo, Kandy)                                   │
│              - to_location: Destination/ending location                                                         │
│              - time: Time preference (e.g., '6 pm', 'afternoon', 'morning')                                     │
│              - date: Date if mentioned (e.g., 'tomorrow', 'today', specific date)                               │
│              Return ONLY valid JSON: {"from_location": "<location>", "to_location": "<location>",               │
│              ""time": "<time>", "date": "<date>"}                                                               │
│              Use empty string '' for missing values.                                                            │
│              You MUST return ONLY valid JSON, no markdown, no explanations.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Transport Assistant                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"from_location": "", "to_location": "", "time": "", "date": ""}                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Scope Guard                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│  User Query:                                                                                                    │
│  hi                                                                                                             │
│                                                                                                                 │
│  Respond with a polite message explaining that SmartMove only provides                                          │
│  bus schedule and fare information in Sri Lanka.                                                                │
│                                                                                                                 │
│  Response format:                                                                                               │
│                                                                                                                 │
│  "Sorry, I am not aware about 'hi'. SmartMove currently provides information about bus schedules and fares in   │
│  Sri Lanka."                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SmartMove Scope Guard                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Sorry, I am not aware about 'hi'. SmartMove currently provides information about bus schedules and fares in    │
│  Sri Lanka.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sorry, I am not aware about 'hi'. SmartMove currently provides information about bus schedules and fares in Sri Lanka.
